In [1]:
pip install sentence-transformers faiss-cpu wikipedia-api numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.0
    Uninstalling click-8.4.0:
      Successfully uninstalled click-8.4.0


In [ ]:
import urllib.request
import json

class SimpleWikiSearch:
    def __init__(self):
        # This list will hold our dictionary of paragraphs
        self.documents = []

    def fetch_wikipedia_page(self, topic):
        """Fetches raw text from Wikipedia using its official web web API."""
        print(f"🌐 Searching Wikipedia for '{topic}'...")

        # Clean the topic name for a web link (replace spaces with %20)
        formatted_topic = topic.replace(" ", "%20")

        # The URL to request data from Wikipedia's backend API
        url = f"https://en.wikipedia.org/w/api.php?action=query&prop=extracts&exintro=0&explaintext=1&titles={formatted_topic}&format=json"

        try:
            # Send a request to the website
            headers = {'User-Agent': 'SimpleBot/1.0 (beginner_project)'}
            req = urllib.request.Request(url, headers=headers)

            with urllib.request.urlopen(req) as response:
                data = json.loads(response.read().decode())

            # Dig into the Wikipedia JSON data to grab the actual text pages
            pages = data['query']['pages']
            page_id = list(pages.keys())[0]

            if page_id == "-1":
                print("❌ Could not find that topic. Try checking your spelling!")
                return False

            full_text = pages[page_id]['extract']

            # Split the giant text wall into separate paragraphs
            raw_paragraphs = full_text.split("\n")

            # Clean up the paragraphs and filter out empty lines or short headers
            self.documents = [p.strip() for p in raw_paragraphs if len(p.strip()) > 50]

            print(f"✅ Successfully loaded {len(self.documents)} paragraphs from Wikipedia!")
            return True

        except Exception as e:
            print(f"❌ An error occurred while fetching data: {e}")
            return False

    def search(self, query):
        """Searches through our stored paragraphs for matching words."""
        # Convert user query to lowercase so matching isn't picky about capital letters
        query_words = query.lower().split()
        search_results = []

        for paragraph in self.documents:
            lower_paragraph = paragraph.lower()
            score = 0

            # Count how many of the query words appear in this paragraph
            for word in query_words:
                if word in lower_paragraph:
                    score += 1

            # If the paragraph contains at least one of our search words, keep it!
            if score > 0:
                search_results.append({
                    "text": paragraph,
                    "score": score
                })

        # Sort the results so the paragraph with the highest score comes first
        search_results.sort(key=lambda item: item['score'], reverse=True)
        return search_results


# --- Main program loop ---
def main():
    engine = SimpleWikiSearch()

    print("=== 🔍 BEGINNER WIKI RETRIEVER ===")
    topic = input("Enter a Wikipedia topic to download (e.g., Python programming, Space): ").strip()

    if not topic:
        print("Topic cannot be empty. Goodbye!")
        return

    if engine.fetch_wikipedia_page(topic):
        print(f"\n🚀 Database built! You can now search across the topic: '{topic}'")

        while True:
            query = input("\nEnter keywords to search for (or type 'exit' to quit): ").strip()
            if query.lower() == 'exit':
                print("Goodbye!")
                break
            if not query:
                continue

            results = engine.search(query)

            print("\n--- 📋 SEARCH RESULTS ---")
            if not results:
                print("No matching paragraphs found. Try using different keywords!")
            else:
                # Show up to the top 3 best matching results
                for i, match in enumerate(results[:3], 1):
                    print(f"\n[Rank {i}] Word Match Score: {match['score']}")
                    print(f"Content: {match['text']}")
            print("-------------------------")

if __name__ == "__main__":
    main()

=== 🔍 BEGINNER WIKI RETRIEVER ===
